In [1]:
# ============================================================
# FEATURE SELECTION
# Mutual Information + Chi²
# ============================================================
#
# Este script aplica DOS métodos filtro:
#
# 1. Mutual Information
# 2. Chi-cuadrada (χ²)
#
# Objetivo:
# Identificar las variables más importantes del dataset.
#
# ============================================================

In [2]:
# ============================================================
# LIBRERÍAS
# ============================================================

# Manejo de datos
import pandas as pd
import numpy as np

# Métodos de selección de variables
from sklearn.feature_selection import (
    mutual_info_classif,
    chi2,
    SelectKBest
)

# Escalado
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler
)


# ============================================================
# CARGAR DATASET
# ============================================================

# Cargar archivo CSV
df = pd.read_csv("00-train.csv")

# ============================================================
# SEPARAR FEATURES Y TARGET
# ============================================================

# IMPORTANTE:
# Quitar los indicadores alfabetivos en este caso "Class" 
# y los demas indicadores alfabetivos, 
# ya que no se pueden usar en los métodos de selección de variables.

X = df.drop(columns=["Class"])
y = df["Class"]

print("\n===================================")
print("DIMENSIONES")
print("===================================")

print("X:", X.shape)
print("y:", y.shape)



DIMENSIONES
X: (198, 95)
y: (198,)


In [4]:
# ============================================================
# MÉTODO 1 — MUTUAL INFORMATION
# ============================================================

print("\n\n===================================")
print("MUTUAL INFORMATION")
print("===================================")


# ============================================================
# ESCALADO PARA MUTUAL INFORMATION
# ============================================================

# se hace para normalizar los valores 
# y evitar que las variables con rangos más grandes dominen la selección

scaler_mi = StandardScaler()

X_scaled_mi = scaler_mi.fit_transform(X)

print("\nDatos escalados correctamente.")


# ============================================================
# CALCULAR MUTUAL INFORMATION
# ============================================================

# Calcula cuánta información aporta cada variable

mi_scores = mutual_info_classif(

    # Variables predictoras (features)
    X_scaled_mi,

    # Variable objetivo (target / clase)
    y,

    # Semilla aleatoria para reproducibilidad
    random_state=42
)

print("Mutual Information calculada correctamente.")


# ============================================================
# CREAR TABLA DE RESULTADOS
# ============================================================

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "MI Score": mi_scores
})

# Ordenar de mayor a menor
mi_df = mi_df.sort_values(
    by="MI Score",
    ascending=False
)

mi_df = mi_df.reset_index(drop=True)


# ============================================================
# SELECCIONAR TOP VARIABLES
# ============================================================

TOP_K = 10

top_features_mi = mi_df["Feature"].head(TOP_K).tolist()

print(f"\nTop {TOP_K} variables (MI):")
print(top_features_mi)


# ============================================================
# DATASET REDUCIDO
# ============================================================

X_mi = X[top_features_mi]

print("\nDimensiones dataset reducido (MI):")
print(X_mi.shape)





MUTUAL INFORMATION

Datos escalados correctamente.
Mutual Information calculada correctamente.

Top 10 variables (MI):
['TS[3]_Q1_T_eps', 'TS[4]_GOWAWA[0.2;2;S-OWA;0.6;0.0;2;S-OWA;0.8;0.1]_T_cch', 'TS[2]_Q1_T_eps', 'TS[4]_V_U_z3', 'CHOQUET[A;-0.75;AO1;1.0]_F_khh', 'TS[3]_AM_F_z1', 'CHOQUET[A;-0.5;AO2;0.0]_N_isa', 'GV[2]_CHOQUET[A;-0.75;AO1;0.3]_H_pah', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_scm', 'TS[3]_CHOQUET[A;-0.75;AO2;0.0]_N_scm']

Dimensiones dataset reducido (MI):
(198, 10)


In [5]:
# ============================================================
# RANDOM FOREST — MUTUAL INFORMATION
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Crear modelo Random Forest
rf_mi = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Entrenar y evaluar usando cross validation
scores_mi = cross_val_score(

    # Modelo
    rf_mi,

    # Dataset reducido por Mutual Information
    X_mi,

    # Clases
    y,

    # Validación cruzada
    cv=5,

    # Métrica
    scoring='f1_weighted'
)

# Mostrar resultados
print("===================================")
print("RANDOM FOREST — MUTUAL INFORMATION")
print("===================================")

print("Scores por fold:")
print(scores_mi)

print("\nF1 promedio:")
print(scores_mi.mean())

RANDOM FOREST — MUTUAL INFORMATION
Scores por fold:
[0.84962406 0.85       0.92457574 0.87162585 0.89606472]

F1 promedio:
0.8783780731018673


In [16]:
# ============================================================
# ============================================================
# MÉTODO 2 — CHI²
# ============================================================
# ============================================================
print("\n\n===================================")
print("CHI²")
print("===================================")

# ============================================================
# ESCALADO PARA CHI²
# ============================================================

# Chi² necesita valores positivos.
# MinMaxScaler transforma variables al rango [0,1]

scaler_chi2 = MinMaxScaler()

X_scaled_chi2 = scaler_chi2.fit_transform(X)

print("\nDatos escalados correctamente para Chi².")


# ============================================================
# SELECCIÓN CHI²
# ============================================================

selector = SelectKBest(
    score_func=chi2,
    k=TOP_K
)

X_chi2 = selector.fit_transform(
    X_scaled_chi2,
    y
)

print("Chi² calculado correctamente.")


# ============================================================
# OBTENER VARIABLES SELECCIONADAS
# ============================================================

selected_mask = selector.get_support()

selected_features_chi2 = X.columns[selected_mask]

chi2_scores = selector.scores_[selected_mask]


# ============================================================
# CREAR TABLA DE RESULTADOS
# ============================================================

chi2_df = pd.DataFrame({
    "Feature": selected_features_chi2,
    "Chi2 Score": chi2_scores
})

chi2_df = chi2_df.sort_values(
    by="Chi2 Score",
    ascending=False
)

chi2_df = chi2_df.reset_index(drop=True)

# ============================================================
# DATASET REDUCIDO
# ============================================================

X_chi2_df = X[selected_features_chi2]

print("\nDimensiones dataset reducido (Chi²):")
print(X_chi2_df.shape)

print("\n Top 10 variables seleccionadas por Chi²:")
print(selected_features_chi2.tolist())





CHI²

Datos escalados correctamente para Chi².
Chi² calculado correctamente.

Dimensiones dataset reducido (Chi²):
(198, 10)

 Top 10 variables seleccionadas por Chi²:
['TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;0;S-OWA;0.0;1.0]_T_cdch', 'TS[4]_V_U_z3', 'GV[2]_CHOQUET[A;-0.75;AO2;0.7]_N_gcp1', 'MIC_GOWAWA[0.0;1;NONE;0.0;0.0;0;S-OWA;0.0;1.0]_N_hwhh', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_H_cch', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_gcp1', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_scm', 'ES_Q3_C_cdch', 'GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_ptt', 'CHOQUET[A;-0.5;AO2;0.0]_N_isa']


In [9]:
# ============================================================
# RANDOM FOREST — CHI²
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Crear modelo Random Forest
rf_chi2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Entrenar y evaluar usando cross validation
scores_chi2 = cross_val_score(

    # Modelo
    rf_chi2,

    # Dataset reducido por Chi²
    X_chi2_df,

    # Clases
    y,

    # Validación cruzada
    cv=5,

    # Métrica
    scoring='f1_weighted'
)

# Mostrar resultados
print("===================================")
print("RANDOM FOREST — CHI²")
print("===================================")

print("Scores por fold:")
print(scores_chi2)

print("\nF1 promedio:")
print(scores_chi2.mean())

RANDOM FOREST — CHI²
Scores por fold:
[0.84848485 0.82401006 0.8989899  0.89730094 0.81361421]

F1 promedio:
0.8564799908455363


In [19]:

# ============================================================
# ============================================================
# COMPARACIÓN ENTRE MÉTODOS
# ============================================================
# ============================================================

print("\n\n===================================")
print("COMPARACIÓN DE VARIABLES")
print("===================================")

print("\nVariables seleccionadas por MI:")
print(top_features_mi)

print("\nVariables seleccionadas por Chi²:")
print(selected_features_chi2.tolist())


# ============================================================
# VARIABLES EN COMÚN
# ============================================================

common_features = list(
    set(top_features_mi).intersection(
        set(selected_features_chi2)
    )
)

print("\nVariables seleccionadas por ambos métodos:")
print(common_features)
print("\nTotal de variables en común:")
print(len(common_features))



COMPARACIÓN DE VARIABLES

Variables seleccionadas por MI:
['TS[3]_Q1_T_eps', 'TS[4]_GOWAWA[0.2;2;S-OWA;0.6;0.0;2;S-OWA;0.8;0.1]_T_cch', 'TS[2]_Q1_T_eps', 'TS[4]_V_U_z3', 'CHOQUET[A;-0.75;AO1;1.0]_F_khh', 'TS[3]_AM_F_z1', 'CHOQUET[A;-0.5;AO2;0.0]_N_isa', 'GV[2]_CHOQUET[A;-0.75;AO1;0.3]_H_pah', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_scm', 'TS[3]_CHOQUET[A;-0.75;AO2;0.0]_N_scm']

Variables seleccionadas por Chi²:
['TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;0;S-OWA;0.0;1.0]_T_cdch', 'TS[4]_V_U_z3', 'GV[2]_CHOQUET[A;-0.75;AO2;0.7]_N_gcp1', 'MIC_GOWAWA[0.0;1;NONE;0.0;0.0;0;S-OWA;0.0;1.0]_N_hwhh', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_H_cch', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_gcp1', 'TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_scm', 'ES_Q3_C_cdch', 'GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_ptt', 'CHOQUET[A;-0.5;AO2;0.0]_N_isa']

Variables seleccionadas por ambos métodos:
['TS[3]_GOWAWA[0.0;1;NONE;0.0;0.0;2;W-OWA;0.5;0.6]_N_scm', 'CHOQUET[A;-0.5;AO